# 02 — Semantic Feature Mapping: Validasi Statistik

Tujuan: membuktikan apakah pasangan fitur kandidat (CIC-IDS2018 ↔ UNSW-NB15) benar-benar **sepadan secara kuantitatif**, bukan sekadar mirip nama. Karena kedua dataset diekstrak dengan alat berbeda (CICFlowMeter vs Argus/Bro), padanan wajib diverifikasi rentang/distribusi/satuan-nya.

**Penting soal skala CIC:** `cleaned_100.pkl` menyimpan fitur yang SUDAH di-`StandardScaler`. Untuk perbandingan yang adil dengan UNSW (raw), notebook ini **mengembalikan skala asli** CIC via `X_orig = X_scaled * scale_ + mean_`.

Output: `mapping_validation.csv` + `mapping_validation.json` (statistik per pasangan + verdict).

> Jalankan di SageMaker (butuh `cleaned_100.pkl`).

In [ ]:
# --- Bootstrap dependency ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json
import numpy as np
import pandas as pd

CIC_PKL   = '../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_CSV  = '../data/UNSW_NB15_testing-set.csv'   # 175.341 record (data latih menurut jumlah)
OUT_CSV   = '../mapping_validation.csv'
OUT_JSON  = '../mapping_validation.json'
print('CIC pkl :', os.path.exists(CIC_PKL))
print('UNSW    :', os.path.exists(UNSW_CSV))

In [ ]:
# --- Muat CIC-IDS2018 & kembalikan ke SKALA ASLI (un-scale) ---
with open(CIC_PKL, 'rb') as f:
    d = pickle.load(f)
cic_feats = list(d['feature_names'])
X = np.asarray(d['X'], dtype=float)
scaler = d.get('scaler', None)

if scaler is not None and hasattr(scaler, 'scale_') and hasattr(scaler, 'mean_'):
    X_orig = X * scaler.scale_ + scaler.mean_
    print('CIC: un-scaled memakai scaler.mean_/scale_')
else:
    X_orig = X
    print('CIC: scaler tidak tersedia -> pakai X apa adanya (hati-hati interpretasi)')

cic_df = pd.DataFrame(X_orig, columns=cic_feats)
print('CIC shape:', cic_df.shape)

In [ ]:
# --- Muat UNSW-NB15 (raw) ---
unsw_df = pd.read_csv(UNSW_CSV)
print('UNSW shape:', unsw_df.shape)

In [ ]:
# --- Pasangan kandidat (hipotesis dari dokumentasi) ---
# (nama CIC, nama UNSW, kategori hipotesis)
pairs = [
    ('Flow Duration',     'dur',    'kuat'),
    ('Tot Fwd Pkts',      'spkts',  'kuat'),
    ('Tot Bwd Pkts',      'dpkts',  'kuat'),
    ('TotLen Fwd Pkts',   'sbytes', 'kuat'),
    ('TotLen Bwd Pkts',   'dbytes', 'kuat'),
    ('Fwd Pkt Len Mean',  'smean',  'kuat'),
    ('Bwd Pkt Len Mean',  'dmean',  'kuat'),
    ('Init Fwd Win Byts', 'swin',   'kuat'),
    ('Init Bwd Win Byts', 'dwin',   'kuat'),
    ('Flow Byts/s',       'sload',  'kuat'),
    ('Bwd Pkts/s',        'dload',  'kuat'),
    ('Fwd IAT Mean',      'sinpkt', 'sedang'),
    ('Bwd IAT Mean',      'dinpkt', 'sedang'),
]

def stat(s):
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    q = s.quantile([0.5, 0.99])
    return dict(min=float(s.min()), median=float(q.loc[0.5]), p99=float(q.loc[0.99]),
                max=float(s.max()), mean=float(s.mean()), std=float(s.std()))

rows = []
for cic_c, unsw_c, hyp in pairs:
    if cic_c not in cic_df.columns:
        rows.append(dict(cic=cic_c, unsw=unsw_c, hyp=hyp, note='CIC col MISSING')); continue
    if unsw_c not in unsw_df.columns:
        rows.append(dict(cic=cic_c, unsw=unsw_c, hyp=hyp, note='UNSW col MISSING')); continue
    cs, us = stat(cic_df[cic_c]), stat(unsw_df[unsw_c])
    # rasio median & p99 sebagai indikator kesetaraan skala
    def ratio(a, b):
        return float(a / b) if b not in (0, 0.0) else float('inf')
    rows.append(dict(
        cic=cic_c, unsw=unsw_c, hyp=hyp,
        cic_median=round(cs['median'], 4), unsw_median=round(us['median'], 4),
        cic_p99=round(cs['p99'], 2), unsw_p99=round(us['p99'], 2),
        cic_max=round(cs['max'], 2), unsw_max=round(us['max'], 2),
        ratio_median=round(ratio(cs['median'], us['median']), 3),
        ratio_p99=round(ratio(cs['p99'], us['p99']), 3),
        note=''))

res = pd.DataFrame(rows)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
print(res.to_string(index=False))

In [ ]:
# --- Verdict otomatis berdasarkan kedekatan skala (median & p99) ---
# Aturan sederhana: jika rasio median & p99 berada di orde yang sama (0.1x..10x) -> 'aligned';
# jika berbeda 1-3 orde -> 'scale-mismatch (perlu konversi)'; >3 orde -> 'likely-different-feature'.
def verdict(r):
    if 'note' in r and r['note'] and 'MISSING' in str(r['note']):
        return 'missing'
    rat = r.get('ratio_p99', None)
    if rat is None or rat != rat or rat in (float('inf'),):
        return 'check-manual'
    a = abs(np.log10(rat)) if rat > 0 else 99
    if a <= 1:   return 'aligned'
    if a <= 3:   return 'scale-mismatch (perlu konversi/scaling)'
    return 'likely-different-feature'

res['verdict'] = res.apply(verdict, axis=1)
print(res[['cic','unsw','hyp','ratio_median','ratio_p99','verdict']].to_string(index=False))

res.to_csv(OUT_CSV, index=False)
res.to_json(OUT_JSON, orient='records', indent=2)
print('\nSaved:', OUT_CSV, '&', OUT_JSON)
print('\nCatatan: verdict ini indikator awal berbasis skala; konfirmasi akhir tetap perlu penalaran domain (satuan, definisi extractor).')

In [ ]:
# Upload hasil validasi pemetaan ke S3 agar bisa diunduh untuk mengisi Tabel SFM di paper.
import os
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for f in [OUT_CSV, OUT_JSON]:
        if os.path.exists(f):
            s3.upload_file(f, S3_BUCKET, f"{S3_PREFIX}/mapping/{os.path.basename(f)}"); up+=1; print('  upload',f)
    print(f'Upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/mapping/')
except Exception as e:
    print('upload S3 dilewati:', e)
print('=== SEL UPLOAD (mapping validation -> S3) SELESAI ===')

## Kuantifikasi Kompleksitas Domain (entropi & variansi 9 fitur SFM)

Membuktikan secara kuantitatif bahwa UNSW = domain lebih kaya/kompleks daripada CIC.
Untuk tiap dari 9 fitur SFM, hitung: (i) variansi pada skala ter-standarisasi global
(z-score gabungan agar setara), dan (ii) entropi Shannon histogram (log1p, 30 bin).
Rata-rata antar-9-fitur dibandingkan CIC vs UNSW. Upload ke S3 unsw-far/mapping/.

In [ ]:
import numpy as np, pandas as pd, json, os
CANON9=['Flow Duration','Tot Fwd Pkts','Tot Bwd Pkts','TotLen Fwd Pkts','TotLen Bwd Pkts',
        'Fwd Pkt Len Mean','Bwd Pkt Len Mean','Flow Byts/s','Bwd Pkts/s']
UNS9 =['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload']
# cic_df (skala asli) & unsw_df sudah tersedia dari sel sebelumnya di notebook ini.
cic9 = cic_df[CANON9].apply(pd.to_numeric, errors='coerce')
uns9 = unsw_df[UNS9].apply(pd.to_numeric, errors='coerce'); uns9.columns=CANON9
cic9=cic9.replace([np.inf,-np.inf],np.nan).dropna(); uns9=uns9.replace([np.inf,-np.inf],np.nan).dropna()

def shannon_entropy(x, bins=30):
    x=np.log1p(np.clip(np.asarray(x,float),0,None))
    h,_=np.histogram(x,bins=bins,density=False); p=h/max(1,h.sum()); p=p[p>0]
    return float(-(p*np.log2(p)).sum())

# Variansi pada z-space GABUNGAN (fit mean/std pada gabungan agar skala setara antar-domain)
both=pd.concat([cic9,uns9],ignore_index=True); mu=both.mean(); sd=both.std().replace(0,1)
zc=(cic9-mu)/sd; zu=(uns9-mu)/sd
rows=[]
for f in CANON9:
    rows.append({'fitur':f,
                 'var_cic':round(float(zc[f].var()),4),'var_uns':round(float(zu[f].var()),4),
                 'ent_cic':round(shannon_entropy(cic9[f]),4),'ent_uns':round(shannon_entropy(uns9[f]),4)})
dfx=pd.DataFrame(rows)
summary={'mean_var_cic':round(float(dfx['var_cic'].mean()),4),'mean_var_uns':round(float(dfx['var_uns'].mean()),4),
         'mean_ent_cic':round(float(dfx['ent_cic'].mean()),4),'mean_ent_uns':round(float(dfx['ent_uns'].mean()),4)}
print('Per-fitur:'); 
import IPython.display as ipd; ipd.display(dfx)
print('Ringkas (rata-rata 9 fitur):', json.dumps(summary,indent=2))
out={'per_feature':rows,'summary':summary,'n_cic':int(len(cic9)),'n_uns':int(len(uns9))}
json.dump(out, open('domain_complexity.json','w'), indent=2)
try:
    import boto3; s3=boto3.client('s3',region_name=os.environ.get('AWS_REGION','ap-southeast-1'))
    s3.upload_file('domain_complexity.json','ssh-detection-features-232032302717','unsw-far/mapping/domain_complexity.json')
    print('upload -> s3://.../unsw-far/mapping/domain_complexity.json')
except Exception as e: print('upload dilewati:',e)
print('=== SEL KOMPLEKSITAS DOMAIN SELESAI ===')